### LCEL Deepdive

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)

In [12]:
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")
model = chat_model
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"topic": "ice cream"})

'Why did the ice cream go to the party? Because it was feeling a little "chill"ed!'

In [13]:
prompt.invoke({"topic": "ice cream"})

ChatPromptValue(messages=[HumanMessage(content='tell me a short joke about ice cream', additional_kwargs={}, response_metadata={})])

In [14]:
from langchain_core.messages.human import HumanMessage

messages = [HumanMessage(content='tell me a short joke about ice cream')]
model.invoke(messages)

AIMessage(content='Why did the ice cream go to the party? Because it was cool with everyone!', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 17, 'completion_tokens': 17, 'total_tokens': 34}, 'model_name': 'gpt-4.1-nano', 'system_fingerprint': None, 'finish_reason': 'stop', 'model': 'gpt-4.1-nano', 'created': 1773132919}, id='lc_run--019cd6f5-30c2-76a0-886e-da3887d2f094-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 17, 'total_tokens': 34})

### Runnables from LangChain

In [15]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [16]:
chain = RunnablePassthrough()
chain.invoke("hello")

'hello'

In [17]:
chain = RunnablePassthrough() | RunnablePassthrough () | RunnablePassthrough ()
chain.invoke("hello")

'hello'

In [18]:
def input_to_upper(input: str):
    output = input.upper()
    return output

In [19]:
chain = RunnableLambda(input_to_upper)
chain.invoke("hello")

'HELLO'

In [20]:
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

'HELLO'

In [23]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough(), 
        "y": RunnablePassthrough()
    }
)
print(chain.invoke("hello"))
print("-------------------------------------------------------")
print(chain.invoke({"input": "hello", "input2": "goodbye"}))

{'x': 'hello', 'y': 'hello'}
-------------------------------------------------------
{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': {'input': 'hello', 'input2': 'goodbye'}}


In [25]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough(), 
        "y": RunnableLambda(lambda z: z["input2"])
    }
)

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': 'goodbye'}

### Nested chains - now it gets more complicated!

In [26]:
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

chain = RunnableParallel(
    {"x": RunnablePassthrough() | RunnableLambda(find_keys_to_uppercase), 
     "y": RunnableLambda(lambda z: z["input2"])})

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': 'HELLO', 'y': 'goodbye'}

In [28]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough()
    }
)

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}}

In [29]:
def assign_func(_):
    return 100

def multiply(input):
    return input * 10

chain = RunnableParallel(
    {
        "x": RunnablePassthrough()}).assign(extra=RunnableLambda(assign_func))

result = chain.invoke({"input": "hello", "input2": "goodbye"})
print(result)

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'extra': 100}


### Combine multiple chains

In [30]:
def extractor(input: dict):
    return input.get("extra", "Key not found")

def cupper(upper: str):
    return str(upper).upper()

new_chain = RunnableLambda(extractor) | RunnableLambda(cupper)

new_chain.invoke({"extra": "test"})

'TEST'

In [31]:
final_chain = chain | new_chain
final_chain.invoke({"input": "hello", "input2": "goodbye"})

'100'